In [1]:
import sys
sys.path.append("../")

In [2]:
import torch
from qmpsqsc.models import mpsqsc
from qmpsqsc.models import qmps
from importlib import reload

reload(mpsqsc)

<module 'qmpsqsc.models.mpsqsc' from '/Users/keisuke/Documents/projects/mps4qsc/notebooks/../qmpsqsc/models/mpsqsc/__init__.py'>

In [3]:
def flip_sites_in_mps(mps_state : mpsqsc.MPState, site_indices):
    """
    In-place flip of given sites in an MPS.

    Assumes:
        - mps_state[site] is a torch.Tensor
        - physical index is dimension 1 and has size 2
    """
    mps_state = mps_state.copy()
    As = mps_state.As
    X = torch.tensor([[0, 1], [1, 0]], device=As[0].device, dtype=As[0].dtype)
    for i in site_indices:
        A = mps_state.As[i]
        if i != 0 and i != L - 1:
            A = torch.einsum("iaj, ab -> ibj", A, X)
        elif i == 0:
            A = torch.einsum("aj, ab -> bj", A, X)
        else:  # ind == L - 1
            A = torch.einsum("ia, ab -> ib", A, X)
        mps_state.As[i] = A
    return mps_state

In [4]:
import torch.nn.functional as F

L = 30
chi = 2
d = 2
ghz = mpsqsc.build_ghz_state(L, d, chi)
ghz = ghz.normalize()

ghz2 = ghz.copy()
ghz2.As[0][:, 1] = -ghz2.As[0][:, 1]

ghz_errors = [flip_sites_in_mps(ghz, [i]) for i in range(L)]
for ghz_error in ghz_errors:
    ghz_error.normalize()

ghz2_errors = [flip_sites_in_mps(ghz2, [i]) for i in range(L)]
for ghz2_error in ghz2_errors:
    ghz2_error.normalize()

ghzs1 = mpsqsc.add_mpstates([ghz] + ghz_errors)
ghzs2 = mpsqsc.add_mpstates([ghz2] + ghz2_errors)
ghzs1 = ghzs1.normalize()
ghzs2 = ghzs2.normalize()


allup = mpsqsc.build_classical_state(L, d, [0]*L)
alldown = mpsqsc.build_classical_state(L, d, [1]*L)

allup_errors = [flip_sites_in_mps(allup, [i]) for i in range(L)]
alldown_errors = [flip_sites_in_mps(alldown, [i]) for i in range(L)]

mixed_states = mpsqsc.add_mpstates([allup, alldown] + allup_errors + alldown_errors)
mixed_states = mixed_states.normalize()





In [5]:
ghzs1.overlap(ghz), ghzs2.overlap(ghz)

(tensor(0.1796, dtype=torch.float64), tensor(0., dtype=torch.float64))

In [6]:
from typing import List, Tuple, Iterator

# -------------------------------------------------------------------------
# 1. function to create a dataset (batch) -- now yields batches indefinitely
# -------------------------------------------------------------------------

def mps_binary_predict(mps1, mps2, states):
    # The best way is to create a list of Tensor pairs and stack them along a new dimension
    amps = torch.stack([
        torch.stack([mps1.overlap(s), mps2.overlap(s)])
        for s in states
    ], dim=0)
    norms = amps.norm(dim=-1)
    amps = amps / norms[:, None]
    probs = amps**2
    return probs, norms

def calculate_loss(mps1, mps2, states, labels):
    preds, norms = mps_binary_predict(mps1, mps2, states)
    probs = preds[torch.arange(len(labels)), labels]
    acc = (probs > 0.5).float()
    loss = -torch.log(probs).mean()
    return loss, acc.mean()

@torch.no_grad()
def create_ghz_rho_batch_qsc(
    mpsghz,
    mps_allup,
    mps_alldown,
    batch_size: int,
    error_rate: float,
) -> Iterator[Tuple[List, torch.Tensor]]:
    """
    Create an infinite generator of training batches for GHZ vs rho with local bit-flip errors.

    States composition:
        - 50% GHZ
        - 25% all-up
        - 25% all-down

    For each sample:
        1. Draw C ~ Poisson(L * error_rate) (L = number of sites).
        2. Choose C distinct sites uniformly at random.
        3. Flip the local tensor at those sites (Pauli-X in physical dimension).

    Yields:
        states: list of length batch_size, each an MPS-like object
        labels: LongTensor of shape (batch_size,) on `device`
                0 -> GHZ   (entangled)
                1 -> rho   (product states: all-up / all-down)
    """

    def _flip_sites_in_mps(mps_state, site_indices):
        """
        In-place flip of given sites in an MPS.

        Assumes:
            - mps_state[site] is a torch.Tensor
            - physical index is dimension 1 and has size 2
        """
        As = mps_state.As
        X = torch.tensor([[0, 1], [1, 0]], device=As[0].device, dtype=As[0].dtype)
        L = len(mps_state.As)
        for i in site_indices:
            A = mps_state.As[i]
            if i != 0 and i != L - 1:
                A.data[:] = torch.einsum("iaj, ab -> ibj", A.data, X)
            elif i == 0:
                A.data[:] = torch.einsum("aj, ab -> bj", A.data, X)
            else:  # ind == L - 1
                A.data[:] = torch.einsum("ia, ab -> ib", A.data, X)
            mps_state.As[i] = A

    device = mpsghz.device
    num_sites = mpsghz.L

    while True:
        # ---- Build the desired mixture: 50% GHZ, 25% all-up, 25% all-down ----
        num_ghz = batch_size // 2
        remaining = batch_size - num_ghz
        num_allup = remaining // 2
        num_alldown = remaining - num_allup

        # 0 = GHZ, 1 = all-up, 2 = all-down (internal codes)
        type_codes = (
            [0] * num_ghz +
            [1] * num_allup +
            [2] * num_alldown
        )
        type_codes = torch.tensor(type_codes, device=device)

        # Shuffle to avoid any ordering bias
        perm = torch.randperm(batch_size, device=device)
        type_codes = type_codes[perm]

        states: List = []
        labels: List[int] = []
        errors: List[int] = [] # 0: no error, 1: error

        for idx in range(batch_size):
            code = int(type_codes[idx].item())

            if code == 0:
                base_state = mpsghz
                label = 0  # GHZ
            elif code == 1:
                base_state = mps_allup
                label = 1  # product
            else:  # code == 2
                base_state = mps_alldown
                label = 1  # product

            state = base_state.copy()
            # Randomly flip a site with probability error_rate for this sample
            if torch.rand(1).item() < error_rate:
                errors.append(1)
                site = torch.randint(0, num_sites, (1,)).item()
                _flip_sites_in_mps(state, [site])
            else:
                errors.append(0)
            states.append(state)
            labels.append(label)

        labels_tensor = torch.tensor(labels, dtype=torch.long, device=device)
        yield states, labels_tensor, errors

In [67]:
data_generator = create_ghz_rho_batch_qsc(ghz, allup, alldown, 2**5, 0.5)

In [68]:
states, labels, errors = next(data_generator)
preds, norms = mps_binary_predict(ghzs1, ghzs2, states)

ghzs1.normalize(norm = norms[0].item())
ghzs2.normalize(norm = norms[1].item())


states, labels, errors = next(data_generator)
preds, norms = mps_binary_predict(ghzs1, ghzs2, states)

In [69]:
ghzs1.set_requires_grad(True)
ghzs2.set_requires_grad(True)
optimizer = torch.optim.Adam(ghzs1.As + ghzs2.As, lr=0.001)

optimizer.zero_grad()

for _ in range(1000):
    states, labels, _ = next(data_generator)
    loss, acc = calculate_loss(ghzs1, ghzs2, states, labels)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    print(loss.item(), acc.item())


0.3465735902799724 1.0
0.24835237156452417 1.0
0.0799650378239924 1.0
0.4950491608625395 0.5625
0.8747644439342037 0.5625
0.6711513840909766 0.5625
0.55526773499714 0.84375
0.46606939698692185 0.875
0.628487730373292 0.65625
0.8326766692958156 0.78125
0.7343494049150734 0.75
0.6925438656800832 0.75
0.3834935996114177 0.8125
0.6132649999642139 0.6875
0.7332720197761653 0.71875
0.6022239436308177 0.6875
0.6418480984964742 0.71875
0.665405942158032 0.71875
0.29117358986936664 0.84375
0.4485978784195954 0.8125
0.7044822761333682 0.78125
0.32205657580125957 0.875
0.4396178844794027 0.875
0.3342292719165364 0.875
0.725345472990543 0.78125
0.41465508704429654 0.8125
0.5696891029119779 0.90625
0.5183631926272686 0.84375
0.3174477488970194 0.875
0.33127155224479143 0.90625
0.2583087629164637 0.90625
0.3467954827342477 0.78125
0.3674760919710816 0.8125
0.464169788725285 0.875
0.3388446650651958 0.90625
0.2799941579416141 0.90625
0.3344824831000287 0.84375
0.33732568902850785 0.84375
0.2671206891

KeyboardInterrupt: 

In [73]:
# ghzs1_chi = ghzs1.truncate_bond_dimension(20)
# ghzs2_chi = ghzs2.truncate_bond_dimension(20)
ghzs1_chi = ghzs1_chi.truncate_bond_dimension(10)
ghzs2_chi = ghzs2_chi.truncate_bond_dimension(10)

ghzs1_chi.set_requires_grad(True)
ghzs2_chi.set_requires_grad(True)

optimizer = torch.optim.Adam(ghzs1_chi.As + ghzs2_chi.As, lr=0.0003)

optimizer.zero_grad()

for _ in range(10000):
    states, labels, _ = next(data_generator)
    loss, acc = calculate_loss(ghzs1_chi, ghzs2_chi, states, labels)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    print(loss.item(), acc.item())


0.261190531043228 0.9375
0.42471247686356395 0.8125
0.3160690508562191 0.875
0.4367871749559785 0.75
0.3557106661473791 0.8125
0.3595830395374009 0.8125
0.33750602733183105 0.8125
0.2595988687960701 0.90625
0.3326549388735866 0.875
0.2763140021181309 0.90625
0.3630883776851562 0.78125
0.3160803256094495 0.84375
0.2456538757700802 0.90625
0.2743897988754471 0.90625
0.2367111187765651 0.9375
0.392985878173717 0.78125
0.35234348568308393 0.8125
0.32376630846213783 0.84375
0.3018258515145399 0.84375
0.2852774663581168 0.875
0.3222883674483651 0.8125
0.14545324620654637 1.0
0.3371071931847414 0.875
0.2350340544012222 0.875
0.3518384906728664 0.78125
0.327742645862288 0.78125
0.17673887449162912 0.9375
0.1974693511295454 0.9375
0.16906081963586106 0.90625
0.265102265280024 0.84375
0.2362018969494432 0.875
0.337734144082096 0.78125
0.3790044088583332 0.6875
0.18760909430842282 0.90625
0.23817877971595625 0.84375
0.19092203487544665 0.90625
0.18948059191702057 0.90625
0.2022147043499028 0.9062